# Unit-circle embedding benchmark — sanity checks

Run this notebook after `python -m src.main` to visualise results and inspect the transform geometry.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_json
from src.metrics import format_results_table, delta_vs_baseline

## 1. Load results

In [ ]:
runs = load_json("../outputs/metrics/nomic_results.json")
print(format_results_table(runs))

## 2. Deltas vs native baseline

In [ ]:
deltas = delta_vs_baseline(runs, baseline="native")
for name, d in deltas.items():
    print(f"\n--- {name} ---")
    for k, v in sorted(d.items()):
        sign = "+" if v >= 0 else ""
        print(f"  {k:<40} {sign}{v:.4f}")

## 3. MRR bar chart: all transforms

In [ ]:
names = list(runs.keys())
mrrs  = [runs[n]["retrieval"]["mrr"] for n in names]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(names, mrrs, color=["steelblue" if "uc" in n else "grey" for n in names])
ax.axhline(runs["native"]["retrieval"]["mrr"], color="red", linestyle="--", label="native")
ax.set_ylabel("MRR")
ax.set_title("MRR by transform (blue = unit-circle)")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
os.makedirs("../outputs/plots", exist_ok=True)
plt.savefig("../outputs/plots/mrr_bar.png", dpi=150)
plt.show()

## 4. Neighbor overlap: native vs uc_256 vs pca_256

In [ ]:
keys_of_interest = ["native", "pca_256", "uc_256", "pca_64", "uc_64"]
overlap_key = [k for k in runs["native"]["geometry"] if "overlap" in k][0]

labels = [k for k in keys_of_interest if k in runs]
overlaps = [runs[k]["geometry"][overlap_key] for k in labels]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, overlaps, color=["steelblue" if "uc" in l else "grey" for l in labels])
ax.set_ylabel(overlap_key)
ax.set_title("Neighbor overlap vs native space")
plt.tight_layout()
plt.savefig("../outputs/plots/neighbor_overlap.png", dpi=150)
plt.show()

## 5. Direct geometry check: T1 transform on a toy vector

In [ ]:
from src.transforms import _uc_map, l2_normalize

# Construct a simple 4-dim test vector
x = np.array([[1.0, -1.0, 0.5, -0.5]], dtype=np.float32)
x_norm = l2_normalize(x)
mapped = _uc_map(x_norm, out_dim=4, mode="direct")

print("input (normalised):", x_norm)
print("UC-mapped output  :", mapped)
print("\ncos half         :", np.cos(np.pi * x_norm))
print("sin half         :", np.sin(np.pi * x_norm))

# Visualise on unit circle
thetas = np.pi * x_norm[0]
fig, ax = plt.subplots(figsize=(5, 5))
circle = plt.Circle((0, 0), 1, fill=False, color="lightgrey")
ax.add_patch(circle)
for i, theta in enumerate(thetas):
    ax.plot([0, np.cos(theta)], [0, np.sin(theta)], label=f"x_{i}={x_norm[0,i]:.2f}")
    ax.scatter(np.cos(theta), np.sin(theta), s=80, zorder=5)
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_title("T1 direct angle map")
plt.tight_layout()
plt.savefig("../outputs/plots/uc_geometry.png", dpi=150)
plt.show()

## 6. Success criteria summary

In [ ]:
from src.metrics import check_success_criteria

checks = check_success_criteria(runs, uc_key="uc_256", pca_key="pca_256")
print("Success criteria:")
for criterion, passed in checks.items():
    mark = "PASS" if passed else "FAIL"
    print(f"  [{mark}] {criterion}")